# Risk — Effects + Candidate Selection (top-N)

Gọi lại đúng hàm CLI thật của `packages/risk` (`qshield_risk.cli.effects`,
`qshield_risk.cli.candidates`) — không chứa công thức tài chính riêng, đúng quy tắc CLAUDE.md
("notebook không được chứa core logic — chỉ gọi lại hàm trong `packages/`").

**Luồng:** Effects (CVaR nền + `g`/`c`/`C_ij` cho QUBO) → Candidate selection (xếp hạng + chọn
top-N). Với universe hiện tại (8 mã, `demo_fast`), bước chọn top-N **không lọc gì** — 8 ≤ 10
(`output_candidates`) nên toàn bộ 8 mã đều được chọn, đúng
`configs/profiles/demo_fast.yaml` ("toàn bộ 8 mã được đưa vào risk/quantum demo").

**`artifacts.mode: dev`** — ghi vào `artifacts/dev/risk/`, đè lên mỗi lần chạy lại (giống 2
notebook trước: `data_exploration.ipynb`, `ai_regime_scenarios.ipynb`).

**Điều kiện tiên quyết:** đã chạy `notebooks/exploration/ai_regime_scenarios.ipynb` — cần
`artifacts/dev/scenarios/{stress_scenarios.npz,scenario_manifest.json}` với `gate_status=PASS`.

⚠️ **`configs/risk.yaml` hiện có `transaction_cost.{fee,spread,liquidity_penalty}` và
`weight_sum_tolerance` đều là `null`** (TBD-002, `docs/product/mvp_scope.md` §23 — Phúc đề xuất,
Ngọc duyệt, **chưa có số nào được duyệt**). `qshield_risk.costs.CostRates.from_config()` cố ý
`raise` nếu thiếu — không có default tài chính ngầm. Notebook này dùng **placeholder PROVISIONAL**
(mức phí môi giới VN điển hình, KHÔNG phải số đã duyệt) chỉ để minh họa luồng chạy — mọi output đều
đánh dấu rõ không phải bằng chứng baseline. Xem `plan.md` mục 0.2 / mục "Cần bạn xác nhận".

In [1]:
import os
import time
from pathlib import Path

import pandas as pd
import yaml
from qshield_contracts.config import Config


def _find_project_root(marker: str = "CLAUDE.md") -> Path:
    p = Path.cwd().resolve()
    for candidate in (p, *p.parents):
        if (candidate / marker).exists():
            return candidate
    raise RuntimeError(
        f"Không tìm thấy {marker} từ {p} trở lên — notebook phải nằm trong repo QSHIELD."
    )


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)

CONFIG_PATH = PROJECT_ROOT / "configs" / "base.yaml"
cfg = Config.load(CONFIG_PATH)
cfg["artifacts"]["mode"]

'dev'

> **Vì sao `os.chdir(PROJECT_ROOT)`:** cùng lý do đã giải thích ở `data_exploration.ipynb` —
> `ArtifactPaths` build đường dẫn tương đối theo repo root, không phải cwd mặc định của kernel.

## Bước 0: Kiểm tra tiên quyết + config PROVISIONAL cho transaction cost

In [2]:
required_scenario_files = [
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "stress_scenarios.npz",
    PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "scenario_manifest.json",
]
missing = [p for p in required_scenario_files if not p.exists()]
if missing:
    raise RuntimeError(
        "Thiếu " + ", ".join(str(p) for p in missing) + " — chạy "
        "notebooks/exploration/ai_regime_scenarios.ipynb trước."
    )
print("OK — đủ scenario cube cho risk.")

OK — đủ scenario cube cho risk.


In [3]:
import json

from qshield_contracts.paths import ArtifactPaths

# `num_scenarios`/`horizon_days` PHẢI khớp đúng cube thật đang nằm trên đĩa (artifacts/dev/
# scenarios/), không phải giá trị mặc định trong configs/scenarios.yaml — cube đó có thể đến từ
# `ai_regime_scenarios.ipynb` (5000, tier "final") hoặc `qshield-ai scenarios` thường (500, tier
# dev). Đọc lại đúng từ scenario_manifest.json để không lệch, thay vì đoán.
scenario_manifest = json.loads(
    (
        PROJECT_ROOT / "artifacts" / "dev" / "scenarios" / "scenario_manifest.json"
    ).read_text(encoding="utf-8")
)

# PROVISIONAL — KHÔNG phải số đã duyệt (TBD-002, xem cảnh báo ở đầu notebook). Chỉ để
# `CostRates.from_config()`/`required_float()` không raise, cho luồng chạy được minh họa.
# Mức phí môi giới VN điển hình, không phải số thật của Q-SHIELD.
PROVISIONAL_TRANSACTION_COST = {
    "fee": 0.0015,
    "spread": 0.0010,
    "liquidity_penalty": 0.0005,
}
PROVISIONAL_WEIGHT_SUM_TOLERANCE = 1e-6

resolved_cfg = dict(cfg)
resolved_cfg["transaction_cost"] = PROVISIONAL_TRANSACTION_COST
resolved_cfg["weight_sum_tolerance"] = PROVISIONAL_WEIGHT_SUM_TOLERANCE
resolved_cfg["num_scenarios"] = scenario_manifest["num_scenarios"]
resolved_cfg["horizon_days"] = scenario_manifest["horizon_days"]

paths = ArtifactPaths(resolved_cfg, run_id=None)
paths.run_root.mkdir(parents=True, exist_ok=True)
resolved_config_path = paths.run_root / "_risk_resolved_config.yaml"
resolved_config_path.write_text(
    yaml.safe_dump(resolved_cfg, allow_unicode=True), encoding="utf-8"
)
print(
    f"Config đã resolve (num_scenarios={resolved_cfg['num_scenarios']}, "
    f"transaction_cost PROVISIONAL) → {resolved_config_path}"
)
print("⚠️  NON_BASELINE_RUN — số phí là placeholder, chưa được Phúc/Ngọc duyệt.")

Config đã resolve (num_scenarios=5000, transaction_cost PROVISIONAL) → artifacts/dev/_risk_resolved_config.yaml
⚠️  NON_BASELINE_RUN — số phí là placeholder, chưa được Phúc/Ngọc duyệt.


## Bước 1: Effects (CVaR nền + g/c/C_ij cho QUBO)

Gọi thẳng `qshield_risk.cli.effects()` — CVaR trước hedge trên danh mục mẫu (`sample_portfolio_
weights`), cộng `g_i`/`c_i` (lợi ích/chi phí từng hành động đơn lẻ) và `C_ij` (tương tác cặp) —
đúng công thức CLAUDE.md quy tắc 14 phía Risk.

Output: `artifacts/dev/risk/{baseline_risk.json,action_effects.csv,pairwise_effects.csv}`.

In [4]:
from qshield_risk.cli import effects as risk_effects

RUN_START = time.perf_counter()
risk_effects(config=str(resolved_config_path), mock=False)

[risk] input_source=real actions=8 pairs=28 -> artifacts/dev/risk


In [5]:
import json

baseline = json.loads(
    (PROJECT_ROOT / "artifacts" / "dev" / "risk" / "baseline_risk.json").read_text(
        encoding="utf-8"
    )
)
{"cvar_0": baseline["cvar_0"], "var_0": baseline["var_0"], "alpha": baseline["alpha"]}

{'cvar_0': 0.1109732123107091, 'var_0': 0.08138182119657209, 'alpha': 0.95}

In [6]:
action_effects = pd.read_csv(
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "action_effects.csv"
)
action_effects.sort_values("g", ascending=False)

,action_id,ticker,g,c
1,1,CTG,0.003510,0.000075
3,3,HPG,0.003333,0.000075
5,5,MWG,0.002723,0.000075
2,2,VCB,0.002588,0.000075
7,7,FPT,0.002498,0.000075
0,0,ACB,0.002493,0.000075
4,4,VIC,0.002481,0.000075
6,6,VNM,0.002246,0.000075


## Bước 2: Candidate selection (top-N)

Gọi thẳng `qshield_risk.cli.candidates()` — xếp hạng theo `net_risk_score` (PROVISIONAL: `g - c`,
chưa được Phúc/Ngọc duyệt trọng số), chọn top `output_candidates`
(`configs/risk.yaml: candidate_selection.output_candidates`, hiện = 10).

Với 8 mã hiện tại: kỳ vọng **toàn bộ 8/8 mã `selected_top10=True`**, `reason="N=8 <=
output_candidates=10, không lọc"`.

Output: `artifacts/dev/risk/candidate_top10.csv`.

In [7]:
from qshield_risk.cli import candidates as risk_candidates

risk_candidates(config=str(resolved_config_path))

total_seconds = time.perf_counter() - RUN_START
print(f"[timing] Effects + Candidates: {total_seconds:.1f}s")

[risk] candidates n_tickers=8 selected=8 -> artifacts/dev/risk/candidate_top10.csv
[timing] Effects + Candidates: 0.5s


In [8]:
candidate_top10 = pd.read_csv(
    PROJECT_ROOT / "artifacts" / "dev" / "risk" / "candidate_top10.csv"
)
n_selected = int(candidate_top10["selected_top10"].sum())
print(f"{len(candidate_top10)} mã — {n_selected} được chọn (selected_top10=True)")
candidate_top10

8 mã — 8 được chọn (selected_top10=True)


,rank,ticker,current_weight,eligible_status,baseline_CVaR_contribution,marginal_CVaR_reduction_10pct,marginal_CVaR_reduction_20pct,marginal_CVaR_reduction_30pct,transaction_cost_estimate,liquidity_penalty,net_risk_score,selected_top10,reason,note
0,1,CTG,0.125,eligible,0.013872,NaN,0.003510,NaN,0.000075,0.000013,0.003435,True,"N=8 <= output_candidates=10, không lọc",NaN
1,2,HPG,0.125,eligible,0.013872,NaN,0.003333,NaN,0.000075,0.000013,0.003258,True,"N=8 <= output_candidates=10, không lọc",NaN
2,3,MWG,0.125,eligible,0.013872,NaN,0.002723,NaN,0.000075,0.000013,0.002648,True,"N=8 <= output_candidates=10, không lọc",NaN
3,4,VCB,0.125,eligible,0.013872,NaN,0.002588,NaN,0.000075,0.000013,0.002513,True,"N=8 <= output_candidates=10, không lọc",NaN
4,5,FPT,0.125,eligible,0.013872,NaN,0.002498,NaN,0.000075,0.000013,0.002423,True,"N=8 <= output_candidates=10, không lọc",NaN
5,6,ACB,0.125,eligible,0.013872,NaN,0.002493,NaN,0.000075,0.000013,0.002418,True,"N=8 <= output_candidates=10, không lọc",NaN
6,7,VIC,0.125,eligible,0.013872,NaN,0.002481,NaN,0.000075,0.000013,0.002406,True,"N=8 <= output_candidates=10, không lọc",NaN
7,8,VNM,0.125,eligible,0.013872,NaN,0.002246,NaN,0.000075,0.000013,0.002171,True,"N=8 <= output_candidates=10, không lọc",NaN


## Xong — checklist đầu ra

- `artifacts/dev/risk/{baseline_risk.json,action_effects.csv,pairwise_effects.csv,candidate_top10.csv}`
- `artifacts/dev/{config.json,data_version.json,metrics.json,logs.txt}` — do `RunContext` ghi
  (CLAUDE.md quy tắc 13).
- `artifacts/dev/_risk_resolved_config.yaml` — config thật đã dùng (kèm `transaction_cost`
  PROVISIONAL), giữ lại để đối chiếu/tái lập.

**⚠️ Toàn bộ số liệu trong lần chạy này là `NON_BASELINE_RUN`** — `transaction_cost` là
placeholder minh họa, chưa phải số đã duyệt (TBD-002). Khi Phúc/Ngọc chốt số thật, cập nhật
`configs/risk.yaml` rồi chạy lại `qshield-risk effects`/`candidates` trực tiếp (không cần override
trong notebook nữa) để có kết quả dùng làm baseline.

**Bước tiếp theo (chưa có trong notebook này):** nối sang `packages/quantum`
(`qshield-quantum solve`, không `--mock`) — giờ có thể thử vì `action_effects.csv`/
`pairwise_effects.csv` thật đã tồn tại; và `qshield_risk.evaluate()` (đã có sẵn) có thể tháo được
blocker "chấm lại true CVaR" ở `packages/quantum/cli.py` — việc riêng, chưa làm ở đây.